# 03 · Diagnostic de la qualité des données

**Projet 4 : Analyse des données de vente et retours** · Sprint 2 · Livrable L3 (diagnostic)

**Objectif** : mesurer la qualité des ventes jointes **sans rien modifier**. Ce notebook observe, compte et documente. Les corrections se font dans `04_nettoyage`, une fois que l'équipe a validé les règles.

| Étape | Ce qu'on vérifie | Question posée |
|---|---|---|
| 1. Chargement | La table produite par `02_flocon` | A-t-on bien toutes les lignes ? |
| 2. Structure et types | Type de chaque colonne | Les dates sont-elles des dates, les montants des nombres ? |
| 3. Valeurs manquantes | Cellules vides par colonne | Où manque-t-il de l'information, et pourquoi ? |
| 4. Doublons | Lignes ou identifiants répétés | Une vente est-elle comptée deux fois ? |
| 5. Variables texte | Espaces, casse, colonnes constantes | Une même modalité est-elle écrite de deux façons ? |
| 6. Plages de valeurs | Négatifs, zéros | Y a-t-il des valeurs impossibles ? |
| 7. Règles de gestion | Montant, remise, retour, coût | Les colonnes sont-elles cohérentes entre elles ? |
| 8. Promotions | Période et remise | La promotion était-elle active le jour de la vente ? |
| 9. Temps | Jours couverts, activité des boutiques | Y a-t-il des jours sans vente ? |
| 10. Valeurs extrêmes | Méthode de l'écart interquartile | Les très grandes valeurs sont-elles des erreurs ? |
| 11. Nature des données | Quantités, prix, clients, COVID | Le fichier reflète-t-il une vraie activité commerciale ? |
| 12. Synthèse | Registre des anomalies | Que faut-il décider avant le nettoyage ? |

**Périmètre** : les **ventes complètes** de nos 56 boutiques sur **trois années** (8 catégories de produits, 12 trimestres) : 2020 et 2021, les deux années de l'énoncé, et 2019, l'année de référence ajoutée pour expliquer la situation de 2021. Le changement de périmètre est documenté et justifié dans `00_inventaire`, section 9. Les constats propres à 2019 sont signalés dans le texte : c'est l'année qui porte le plus de lignes (339 057 sur 715 291) et la plupart des anomalies.

**Lit dans** : `data/interim/ventes_jointes.parquet`
**Écrit dans** : `reports/livrables/L3_diagnostic_nettoyage/diagnostic_qualite.xlsx` et `reports/figures`

**Principe du registre** : chaque contrôle appelle la fonction `noter(...)`, qui ajoute une ligne à un registre (contrôle, nombre de lignes concernées, gravité, action proposée). La section 12 affiche ce registre : c'est la base de discussion pour décider des règles de `04_nettoyage`.

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
# On remonte de dossier en dossier jusqu'à trouver celui qui contient src/ et data/.
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent

# Ajouter la racine aux dossiers où Python cherche les modules
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins des dossiers, période attendue (DATE_DEBUT, DATE_FIN)
#   viz    -> charte graphique (mêmes couleurs et mêmes polices dans tout le projet)
#   etl    -> lecture d'une table de dimension brute (utilisée en section 8 pour DimPromotion)
from src import config, etl, viz

config.creer_dossiers()
viz.appliquer_style()

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Racine du projet :", config.ROOT)

# ---------------------------------------------------------------------------
# 3. Tolérance pour comparer des montants
# ---------------------------------------------------------------------------
# Les montants sont arrondis au cent dans les fichiers. Un calcul refait à partir du
# prix et de la quantité peut donc différer d'un cent du montant enregistré.
# On ne compte comme anomalie qu'un écart STRICTEMENT supérieur à cette tolérance.
TOLERANCE = 0.02

# ---------------------------------------------------------------------------
# 4. Registre des anomalies
# ---------------------------------------------------------------------------
# Liste de dictionnaires : une entrée par contrôle effectué.
registre = []


def noter(section, controle, nb, gravite, action, total=None, zero_est_ok=True):
    """Ajoute un contrôle au registre et affiche son résultat.

    section  : numéro et nom de la section du notebook (pour retrouver le contrôle)
    controle : ce qu'on a vérifié, en une phrase
    nb       : nombre de lignes (ou de colonnes, de jours...) concernées
    gravite  : "OK"          -> rien à signaler
               "Information" -> particularité normale, à connaître pour l'analyse
               "À corriger"  -> correction simple et sans ambiguïté dans 04_nettoyage
               "À décider"   -> plusieurs traitements possibles : l'équipe tranche
    action   : traitement proposé pour 04_nettoyage
    total    : dénominateur pour calculer le pourcentage (par défaut : nombre de ventes)
    zero_est_ok : en général, 0 ligne concernée veut dire "rien à signaler" et la gravité devient "OK".
               Mettre False quand c'est justement le zéro qui est le constat (section 11 :
               zéro vente à l'unité, zéro changement de prix).
    """
    nb = int(nb)
    if nb == 0 and zero_est_ok:
        gravite = "OK"
    total = len(ventes) if total is None else total
    registre.append({
        "section": section, "controle": controle, "nb": nb,
        "part": nb / total if total else np.nan, "gravite": gravite, "action": action,
    })
    nb_lisible = f"{nb:,}".replace(",", " ")   # séparateur de milliers : espace
    print(f"[{gravite:<11}] {controle} : {nb_lisible} ({nb / total:.2%})")

## 1. Chargement

In [ ]:
# ---------------------------------------------------------------------------
# Ventes jointes : une ligne par vente, avec les attributs de la boutique, du produit,
# de la promotion, du canal et le contexte du pays ce jour-là (férié, sévérité COVID).
# ---------------------------------------------------------------------------
chemin = config.INTERIM / "ventes_jointes.parquet"
if not chemin.exists():
    raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord 01_etl puis 02_flocon.")

ventes = pd.read_parquet(chemin)

# Les montants ont été enregistrés dans un type décimal "nullable" (Float64, avec F majuscule).
# On les convertit en float64 classique : les calculs numpy et les graphiques sont plus simples.
# Cette conversion ne change aucune valeur et ne touche que la copie en mémoire, pas le fichier.
colonnes_montants = ["prix_unitaire", "cout_unitaire", "montant_ventes", "cout_total",
                     "montant_remise", "montant_retour"]
for col in colonnes_montants:
    ventes[col] = ventes[col].astype("float64")

print(f"Lignes   : {len(ventes):,}".replace(",", " "))
print(f"Colonnes : {ventes.shape[1]}")
print(f"Période  : du {ventes.date.min():%Y-%m-%d} au {ventes.date.max():%Y-%m-%d}")
print(f"Chiffre d'affaires : {ventes.montant_ventes.sum():,.0f} $".replace(",", " "))

# Contrôle de continuité avec 01_etl : le nombre de lignes ne doit pas avoir changé.
# Une jointure mal faite (clé en double dans une dimension) multiplierait les lignes.
consolidees = pd.read_parquet(config.INTERIM / "ventes_consolidees.parquet", columns=["id_vente"])
assert len(ventes) == len(consolidees), "Le nombre de lignes a changé entre 01_etl et 02_flocon."
print("Même nombre de lignes qu'à la sortie de 01_etl : OK")

## 2. Structure et types

Un type incorrect fausse tout le reste : une date lue comme du texte ne se trie pas, un montant lu comme du texte ne s'additionne pas. On regroupe les colonnes par famille pour vérifier chaque famille d'un coup d'œil.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Tableau de structure : type, nombre de valeurs distinctes, exemple
# ---------------------------------------------------------------------------
structure = pd.DataFrame({
    "type": ventes.dtypes.astype(str),
    "valeurs_distinctes": ventes.nunique(),
    "manquantes": ventes.isna().sum(),
    "exemple": ventes.iloc[0].astype(str),
})
display(structure)

# ---------------------------------------------------------------------------
# 2. Vérifier le type de chaque famille de colonnes
# ---------------------------------------------------------------------------
# Les familles sont définies ici une fois et réutilisées dans les sections suivantes.
COL_DATES = ["date", "date_ouverture", "date_fermeture", "promo_date_debut", "promo_date_fin"]
COL_QUANTITES = ["quantite_vente", "quantite_remise", "quantite_retour"]
COL_MONTANTS = colonnes_montants
COL_TEXTE = [c for c in ventes.columns if ventes[c].dtype == object or str(ventes[c].dtype) in ("str", "string")]

mauvaises_dates = [c for c in COL_DATES if not pd.api.types.is_datetime64_any_dtype(ventes[c])]
mauvais_nombres = [c for c in COL_QUANTITES + COL_MONTANTS if not pd.api.types.is_numeric_dtype(ventes[c])]

noter("2. Types", "Colonnes de dates qui ne sont pas de type date", len(mauvaises_dates),
      "À corriger", "Convertir avec pd.to_datetime", total=len(COL_DATES))
noter("2. Types", "Colonnes numériques qui ne sont pas de type nombre", len(mauvais_nombres),
      "À corriger", "Convertir avec etl.convertir_montant", total=len(COL_QUANTITES + COL_MONTANTS))

# ---------------------------------------------------------------------------
# 3. Les dates de vente portent-elles une heure ?
# ---------------------------------------------------------------------------
# Dans les fichiers, la date est écrite "1/2/2020 0:00". Si l'heure est toujours 0:00,
# elle ne porte aucune information : on raisonnera au jour.
avec_heure = (ventes.date != ventes.date.dt.normalize()).sum()
noter("2. Types", "Dates de vente avec une heure différente de 00:00", avec_heure,
      "Information", "Aucune : l'analyse se fait au jour")

## 3. Valeurs manquantes

Une valeur manquante n'est pas forcément une erreur. Il faut distinguer trois cas :

- **manque normal** : l'information n'existe pas (une boutique encore ouverte n'a pas de date de fermeture, un jour ordinaire n'a pas de nom de jour férié) ;
- **manque dans la source** : Contoso n'a pas renseigné la valeur ;
- **manque créé par nous** : une jointure qui n'a pas trouvé de correspondance. C'est le seul cas qui signale une erreur de notre chaîne.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Nombre et part de valeurs manquantes par colonne
# ---------------------------------------------------------------------------
manquantes = (ventes.isna().sum()
              .to_frame("nb")
              .assign(part=lambda d: d.nb / len(ventes))
              .query("nb > 0")
              .sort_values("nb", ascending=False))
print(f"Colonnes avec au moins une valeur manquante : {len(manquantes)} sur {ventes.shape[1]}")
display(manquantes.style.format({"nb": "{:,.0f}", "part": "{:.2%}"}))

# ---------------------------------------------------------------------------
# 2. Colonnes qui ne doivent JAMAIS être vides
# ---------------------------------------------------------------------------
# Ce sont les mesures (quantités, montants), les clés, et les attributs ramenés par les
# jointures de 02_flocon. Un vide ici voudrait dire : vente sans boutique, sans produit...
OBLIGATOIRES = ["id_vente", "date", "id_boutique", "id_produit", "id_promotion", "id_canal",
                *COL_QUANTITES, *COL_MONTANTS,
                "nom_boutique", "pays", "code_pays", "nom_produit", "categorie", "sous_categorie",
                "nom_promotion", "taux_remise_annonce", "canal", "est_ferie", "severite_covid"]
vides_obligatoires = ventes[OBLIGATOIRES].isna().any(axis=1).sum()
noter("3. Manquantes", "Ventes avec un vide dans une colonne obligatoire (mesure, clé, attribut joint)",
      vides_obligatoires, "À décider", "Identifier la jointure en cause dans 02_flocon")

# ---------------------------------------------------------------------------
# 3. Expliquer chaque colonne incomplète
# ---------------------------------------------------------------------------
# a) nom_ferie : vide quand le jour n'est pas férié. On vérifie que les deux colonnes
#    racontent la même chose : nom présent <=> est_ferie vrai.
incoherence_ferie = (ventes.nom_ferie.notna() != ventes.est_ferie).sum()
noter("3. Manquantes", "Incohérence entre nom_ferie et est_ferie", incoherence_ferie,
      "À corriger", "Recalculer est_ferie à partir de nom_ferie")
noter("3. Manquantes", "nom_ferie vide (jour non férié)", ventes.nom_ferie.isna().sum(),
      "Information", "Aucune : manque normal, la colonne est_ferie suffit pour l'analyse")

# b) date_fermeture : vide tant que la boutique est ouverte.
noter("3. Manquantes", "date_fermeture vide (boutique toujours ouverte)", ventes.date_fermeture.isna().sum(),
      "Information", "Supprimer la colonne si elle est entièrement vide")

# c) effectif : quelles boutiques sont concernées ?
sans_effectif = (ventes[ventes.effectif.isna()]
                 .groupby(["id_boutique", "nom_boutique", "type_boutique", "pays"])
                 .size().rename("nb_ventes").reset_index())
print("\nBoutiques sans effectif renseigné :")
display(sans_effectif)
# Attention à la lecture : il ne manque AUCUNE vente. Il manque un seul chiffre, le nombre d'employés
# de la boutique, dans la table des boutiques. Comme il est recopié sur chaque vente de la boutique,
# il apparaît vide autant de fois que la boutique a de lignes.
# La boutique est la même sur les trois années : le nombre de lignes concernées a augmenté avec 2019,
# pas le nombre de boutiques.
noter("3. Manquantes", f"effectif vide ({len(sans_effectif)} boutique(s), manque dans la source Contoso)",
      ventes.effectif.isna().sum(), "À décider",
      "Imputer par la médiane des magasins comparables et ajouter un indicateur effectif_impute (règle N7 de 04_nettoyage)")

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : part de valeurs manquantes par colonne (colonnes incomplètes seulement)
# ---------------------------------------------------------------------------
if len(manquantes):
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(manquantes) + 1.8))
    donnees = manquantes.sort_values("part")
    ax.barh(donnees.index, donnees.part, color=viz.ALERTE)
    # Étiquette à droite de chaque barre : la part exacte
    for y, part in enumerate(donnees.part):
        ax.text(part + 0.01, y, f"{part:.1%}", va="center", color=viz.GRIS)
    ax.set_xlim(0, 1.12)
    viz.format_pourcent(ax, axe="x")
    viz.titre(ax, "Valeurs manquantes par colonne",
              f"{len(manquantes)} colonnes incomplètes sur {ventes.shape[1]} · {len(ventes):,} ventes".replace(",", " "))
    print("Figure enregistrée :", viz.enregistrer(fig, "03_valeurs_manquantes"))
    plt.show()

## 4. Doublons

Trois niveaux de contrôle, du plus strict au plus large :

1. **ligne entière identique** : la même ligne lue deux fois (par exemple un fichier chargé en double) ;
2. **identifiant de vente répété** : deux lignes différentes avec le même `id_vente` ;
3. **clé métier répétée** : même jour, même boutique, même produit, même promotion, même canal. Ce n'est pas forcément une erreur, mais c'est à connaître pour savoir ce que représente une ligne.

In [ ]:
# 1. Lignes entièrement identiques
noter("4. Doublons", "Lignes entièrement identiques", ventes.duplicated().sum(),
      "À corriger", "Supprimer avec drop_duplicates()")

# 2. Identifiant de vente répété
noter("4. Doublons", "Identifiants de vente répétés", ventes.id_vente.duplicated().sum(),
      "À décider", "Comparer les lignes en conflit avant de choisir laquelle garder")

# 3. Clé métier répétée
CLE_METIER = ["date", "id_boutique", "id_produit", "id_promotion", "id_canal"]
noter("4. Doublons", "Clé métier répétée (jour, boutique, produit, promotion, canal)",
      ventes.duplicated(CLE_METIER).sum(), "Information", "Aucune")

# Conséquence pour l'analyse : si la clé métier est unique, une ligne n'est pas un ticket
# de caisse mais le CUMUL des ventes d'un produit, dans une boutique, un jour donné,
# pour une promotion et un canal. C'est ce qui explique des quantités minimales élevées.
print("\nQuantité vendue par ligne : minimum", ventes.quantite_vente.min(),
      "· médiane", ventes.quantite_vente.median(), "· maximum", ventes.quantite_vente.max())

## 5. Variables texte

Deux écritures d'une même modalité (`"Litware"` et `"Litware "`) créent deux groupes dans un `groupby` et faussent les classements. On cherche les espaces en trop, les différences de casse, et les colonnes qui ne prennent qu'une seule valeur (elles n'apportent rien à l'analyse).

In [ ]:
# ---------------------------------------------------------------------------
# 1. Pour chaque colonne texte : modalités, espaces en trop, variantes de casse
# ---------------------------------------------------------------------------
lignes = []
for col in COL_TEXTE:
    modalites = pd.Series(ventes[col].dropna().unique()).astype(str)
    # Espaces au début, à la fin, ou doubles espaces à l'intérieur
    avec_espaces = modalites[(modalites != modalites.str.strip()) | modalites.str.contains("  ")]
    # Modalités qui deviennent identiques une fois nettoyées (espaces retirés, minuscules)
    normalisees = modalites.str.strip().str.lower()
    lignes.append({
        "colonne": col,
        "modalites": len(modalites),
        "avec_espaces": len(avec_espaces),
        "variantes_fusionnables": len(modalites) - normalisees.nunique(),
        "exemples_a_corriger": ", ".join(repr(m) for m in avec_espaces[:4]),
    })
texte = pd.DataFrame(lignes).set_index("colonne")
display(texte)

# ---------------------------------------------------------------------------
# 2. Nombre de ventes concernées par une modalité avec espace en trop
# ---------------------------------------------------------------------------
for col in texte.index[texte.avec_espaces > 0]:
    valeurs = ventes[col].astype("string")
    # Même définition qu'au point 1 : espace au début, à la fin, ou double espace à l'intérieur
    nb = ((valeurs != valeurs.str.strip()) | valeurs.str.contains("  ")).sum()
    noter("5. Texte", f"{col} : valeurs avec espace en trop ({texte.loc[col, 'exemples_a_corriger']})", nb,
          "À corriger", "Retirer les espaces de début et de fin, réduire les doubles espaces")
if (texte.avec_espaces == 0).all():
    noter("5. Texte", "Valeurs texte avec espace en trop", 0, "OK", "Aucune")

# ---------------------------------------------------------------------------
# 3. Colonnes constantes (une seule valeur sur toutes les lignes)
# ---------------------------------------------------------------------------
constantes = [c for c in ventes.columns if ventes[c].nunique(dropna=True) <= 1]
print("\nColonnes constantes :", {c: ventes[c].dropna().unique().tolist()[:1] for c in constantes})
noter("5. Texte", f"Colonnes constantes ou entièrement vides : {', '.join(constantes) or 'aucune'}",
      len(constantes), "Information", "Retirer de la table d'analyse", total=ventes.shape[1])

# ---------------------------------------------------------------------------
# 4. Coquilles dans les libellés (visibles seulement à la lecture)
# ---------------------------------------------------------------------------
print("\nLibellés des promotions :")
display(ventes.groupby(["id_promotion", "nom_promotion", "taux_remise_annonce"]).size().rename("nb_ventes").reset_index())

**À lire dans le tableau des promotions** : le même nom de promotion revient avec plusieurs identifiants (une édition par année), d'où l'importance de garder `id_promotion`. Des promotions nommées *North America* apparaissent sur des boutiques européennes, et le libellé *Back-to-Scholl* contient une coquille d'origine. Ce sont des particularités de Contoso : on les signale, on ne les invente pas.

## 6. Plages de valeurs

On cherche les valeurs **impossibles** (quantité ou montant négatif, prix nul) avant de chercher les valeurs **inhabituelles** (section 10).

In [ ]:
# ---------------------------------------------------------------------------
# 1. Statistiques descriptives des mesures
# ---------------------------------------------------------------------------
display(ventes[COL_QUANTITES + COL_MONTANTS].describe().T)

# ---------------------------------------------------------------------------
# 2. Valeurs négatives : impossibles pour toutes les mesures
# ---------------------------------------------------------------------------
for col in COL_QUANTITES + COL_MONTANTS:
    noter("6. Plages", f"{col} négatif", (ventes[col] < 0).sum(), "À décider",
          "Vérifier s'il s'agit d'un avoir ; sinon exclure la ligne")

# ---------------------------------------------------------------------------
# 3. Zéros : impossibles pour une vente, NORMAUX pour une remise ou un retour
# ---------------------------------------------------------------------------
for col in ["quantite_vente", "prix_unitaire", "cout_unitaire", "montant_ventes", "cout_total"]:
    noter("6. Plages", f"{col} égal à zéro", (ventes[col] == 0).sum(), "À décider",
          "Exclure : une vente sans quantité, sans prix ou sans coût n'est pas exploitable")

# Pour les remises et les retours, le zéro est la situation la plus fréquente.
print("\nPart des ventes sans remise :", f"{(ventes.quantite_remise == 0).mean():.1%}")
print("Part des ventes sans retour :", f"{(ventes.quantite_retour == 0).mean():.1%}")

# ---------------------------------------------------------------------------
# 4. Taux de remise annoncé : doit être compris entre 0 et 1
# ---------------------------------------------------------------------------
hors_bornes = (~ventes.taux_remise_annonce.between(0, 1)).sum()
noter("6. Plages", "Taux de remise annoncé hors de l'intervalle [0 ; 1]", hors_bornes,
      "À corriger", "Diviser par 100 si le taux est exprimé en pourcentage")

## 7. Règles de gestion

Les colonnes d'une vente sont liées par des formules. Si une formule n'est pas respectée, une des colonnes est fausse. On recalcule chaque colonne à partir des autres et on compare au cent près.

| Règle | Formule vérifiée |
|---|---|
| R1 | Montant des ventes = Prix unitaire × Quantité vendue − Montant de la remise |
| R2 | Montant de la remise = Prix unitaire × Quantité remisée × Taux annoncé |
| R3 | Montant du retour = Prix unitaire × Quantité retournée |
| R4 | Coût total = Coût unitaire × Quantité vendue |
| R5 | Quantité retournée ≤ Quantité vendue, Quantité remisée ≤ Quantité vendue |
| R6 | Prix et coût de la vente = prix et coût du catalogue produit |
| R7 | Prix unitaire ≥ Coût unitaire (pas de vente à perte avant remise) |

In [ ]:
# Raccourcis pour alléger l'écriture des formules
q, qr, qret = ventes.quantite_vente, ventes.quantite_remise, ventes.quantite_retour
pu, cu = ventes.prix_unitaire, ventes.cout_unitaire


def ecart(observe, attendu):
    """Vrai pour chaque ligne où la valeur observée s'écarte de la valeur attendue de plus de la tolérance."""
    return (observe - attendu).abs() > TOLERANCE


# R1 : la règle donnée dans l'énoncé du projet
r1 = ecart(ventes.montant_ventes, pu * q - ventes.montant_remise)
noter("7. Règles", "R1 non respectée : montant des ventes ≠ prix × quantité − remise", r1.sum(),
      "À décider", "Recalculer le montant à partir du prix, de la quantité et de la remise")

# Le montant des ventes est-il NET des retours ? On teste l'hypothèse inverse :
# si c'était le cas, la formule avec (quantité − retour) tomberait juste.
r1_net = ecart(ventes.montant_ventes, pu * (q - qret) - ventes.montant_remise)
print(f"   Hypothèse 'montant net des retours' : fausse sur {r1_net.sum():,} lignes".replace(",", " "),
      "-> le montant des ventes est BRUT, les retours sont à soustraire nous-mêmes.")

# R2 : la remise ne s'applique qu'aux unités remisées, au taux de la promotion
r2 = ecart(ventes.montant_remise, pu * qr * ventes.taux_remise_annonce)
noter("7. Règles", "R2 non respectée : remise ≠ prix × quantité remisée × taux annoncé", r2.sum(),
      "À décider", "Garder le montant enregistré et signaler la ligne")

# R3 : un retour est remboursé au prix unitaire plein
r3 = ecart(ventes.montant_retour, pu * qret)
noter("7. Règles", "R3 non respectée : montant du retour ≠ prix × quantité retournée", r3.sum(),
      "À décider", "Recalculer le montant du retour")

# R5 : on ne peut ni retourner ni remiser plus d'unités qu'on en a vendu
noter("7. Règles", "R5 : quantité retournée > quantité vendue", (qret > q).sum(), "À décider", "Plafonner ou exclure")
noter("7. Règles", "R5 : quantité remisée > quantité vendue", (qr > q).sum(), "À décider",
      "Limiter la quantité remisée à la quantité vendue, sans modifier les montants (analyse ci-dessous)")

# R6 : le prix et le coût de la vente sont-ils ceux du catalogue (dim_produit) ?
r6 = ecart(pu, ventes.prix_catalogue) | ecart(cu, ventes.cout_catalogue)
noter("7. Règles", "R6 : prix ou coût de la vente différent du catalogue", r6.sum(),
      "Information", "Aucune")
if r6.sum() == 0:
    print("   Conséquence : le prix de chaque produit est FIXE sur toute la période. La seule variation")
    print("   de prix payée par le client vient de la remise. À retenir pour Q5 (optimisation des prix).")

# R7 : vente à perte avant remise, puis marge en valeur négative après remise
noter("7. Règles", "R7 : prix unitaire inférieur au coût unitaire", (pu < cu).sum(), "À décider", "Vérifier le produit")

### R5 : des remises sur plus d'unités que la vente

La règle R5 n'est pas respectée sur 2 331 lignes (0,33 %) : la quantité remisée dépasse la quantité vendue (par exemple 2 unités vendues, 4 remisées). Ces lignes n'existaient pas dans les fichiers du professeur. Elles viennent de deux sous-catégories seulement : *Download Games* (1 474 lignes, toutes en 2019) et *Washers & Dryers* (471 lignes en 2019, 386 en 2021). Il n'y en a aucune en 2020.

Comme nous n'avons pas d'équipe métier à interroger, on raisonne en trois temps :

1. **Décrire** : où sont ces lignes, et combien pèsent-elles ?
2. **Tester trois explications possibles**, en regardant sur combien d'unités portent le coût, le chiffre d'affaires et la remise :

| Hypothèse | Ce qu'elle signifie | Ce qu'on devrait observer |
|---|---|---|
| H1. Unités offertes | 4 unités sont sorties, dont 2 gratuites | Coût de 4 unités ; remise égale au prix plein des unités offertes |
| H2. Erreur sur la quantité remisée | 2 unités vendues, le « 4 » est faux | Coût et CA de 2 unités ; quantité remisée sans lien avec la quantité vendue |
| H3. Pack (« une achetée, une offerte ») | 2 payées, 2 offertes, coût couvert par le fournisseur | Quantité remisée toujours multiple de la quantité vendue ; même schéma dans les autres boutiques |

3. **Comparer avec les autres boutiques** : pour le même produit, la même promotion et le même mois, comment la remise est-elle enregistrée ailleurs ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Décrire les lignes où la quantité remisée dépasse la quantité vendue
# ---------------------------------------------------------------------------
depassement = qr > q
anomalie = ventes[depassement]
print(f"Lignes concernées : {len(anomalie)} sur {len(ventes):,} ({len(anomalie) / len(ventes):.2%})".replace(",", " "))
if len(anomalie):
    print("Années            :", anomalie.annee.value_counts().to_dict())
    print("Catégories        :", anomalie.categorie.str.strip().value_counts().to_dict())
    print("Sous-catégories   :", anomalie.sous_categorie.str.strip().value_counts().to_dict())
    print("Boutiques touchées:", anomalie.id_boutique.nunique(), "· produits touchés :", anomalie.id_produit.nunique())
    print(f"Prix unitaire     : de {anomalie.prix_unitaire.min():,.2f} $ à {anomalie.prix_unitaire.max():,.2f} $".replace(",", " "))

    # Poids : chiffre d'affaires, remise enregistrée, et remise "en trop"
    # (celle des unités remisées au-delà des unités vendues)
    remise_en_trop = float((anomalie.prix_unitaire * (anomalie.quantite_remise - anomalie.quantite_vente)
                            * anomalie.taux_remise_annonce).sum())
    print(f"\nChiffre d'affaires de ces lignes : {float(anomalie.montant_ventes.sum()):,.0f} $ "
          f"({float(anomalie.montant_ventes.sum()) / float(ventes.montant_ventes.sum()):.2%} du total)".replace(",", " "))
    print(f"Remise enregistrée               : {float(anomalie.montant_remise.sum()):,.0f} $ "
          f"({float(anomalie.montant_remise.sum()) / float(ventes.montant_remise.sum()):.2%} des remises)".replace(",", " "))
    print(f"Dont remise en trop              : {remise_en_trop:,.0f} $ "
          f"({remise_en_trop / float(ventes.montant_remise.sum()):.2%} des remises)".replace(",", " "))

    # -----------------------------------------------------------------------
    # 2. Sur combien d'unités portent le coût, le chiffre d'affaires et la remise ?
    # -----------------------------------------------------------------------
    # On divise chaque montant par son prix unitaire : on obtient le nombre d'unités
    # que le montant représente réellement.
    unites = pd.DataFrame({
        "quantité vendue": anomalie.quantite_vente,
        "quantité remisée": anomalie.quantite_remise,
        "unités couvertes par le coût": (anomalie.cout_total / anomalie.cout_unitaire).round(2),
        "unités couvertes par le CA avant remise": ((anomalie.montant_ventes + anomalie.montant_remise) / anomalie.prix_unitaire).round(2),
        "unités couvertes par la remise": (anomalie.montant_remise / (anomalie.prix_unitaire * anomalie.taux_remise_annonce)).round(2),
    })
    # On ne garde que les lignes sans retour pour cette lecture : sur les lignes avec retour,
    # le coût total suit d'autres logiques (voir R4 plus bas).
    sans_retour = unites[anomalie.quantite_retour == 0]
    print("\nNombre de lignes selon les unités couvertes par chaque montant (lignes sans retour) :")
    display(sans_retour.value_counts().rename("lignes").reset_index().sort_values(["quantité vendue", "quantité remisée"]))
    # Lecture : le coût et le CA avant remise portent sur la quantité VENDUE,
    # la remise porte sur la quantité REMISÉE. H1 demanderait un coût sur plus d'unités
    # et une remise au prix plein : ce n'est pas le cas.

    # -----------------------------------------------------------------------
    # 3. Les mêmes promotions dans les autres boutiques
    # -----------------------------------------------------------------------
    # Pour chaque ligne en anomalie, on prend les ventes du même produit, avec la même
    # promotion, le même mois, qui ne sont PAS en anomalie (donc les autres boutiques ou
    # les autres jours). On regarde comment la quantité remisée s'y répartit.
    mois_vente = ventes.date.dt.to_period("M")
    cas = (anomalie.assign(mois_vente=mois_vente[depassement])[["id_produit", "id_promotion", "mois_vente"]]
           .drop_duplicates())
    comparables = (ventes.assign(mois_vente=mois_vente)[~depassement]
                   .merge(cas, on=["id_produit", "id_promotion", "mois_vente"]))
    print(f"Lignes comparables (même produit, même promotion, même mois, sans anomalie) : {len(comparables)}")
    r2_comparables = ((comparables.montant_remise - comparables.prix_unitaire * comparables.quantite_remise
                       * comparables.taux_remise_annonce).abs() > TOLERANCE).sum()
    print("Lignes comparables où la remise ne vaut pas prix × quantité remisée × taux annoncé :", int(r2_comparables))
    print("\nNombre de lignes comparables selon la quantité vendue (lignes) et la quantité remisée (colonnes) :")
    display(pd.crosstab(comparables.quantite_vente.rename("quantité vendue"),
                        comparables.quantite_remise.rename("quantité remisée")).head(6))
    print("Plus grande part d'unités remisées sur une ligne comparable :",
          f"{float((comparables.quantite_remise / comparables.quantite_vente).max()):.0%}")

    # -----------------------------------------------------------------------
    # 4. Pourquoi certaines années seulement ? La quantité remisée ne dépend pas de la quantité vendue
    # -----------------------------------------------------------------------
    # Pour chaque sous-catégorie touchée et chaque année : la plus petite quantité vendue par ligne,
    # la plus grande quantité remisée, et le nombre de lignes en dépassement.
    # Si le dépassement n'apparaît que les années où le produit se vend par 2 ou 3, c'est que la
    # quantité remisée est tirée sans tenir compte de la quantité vendue.
    for sous_cat in anomalie.sous_categorie.unique():
        meme_famille = ventes[(ventes.sous_categorie == sous_cat) & (ventes.id_promotion != 1)]
        resume = meme_famille.groupby("annee").agg(
            lignes_en_promotion=("id_vente", "size"),
            quantite_vendue_minimale=("quantite_vente", "min"),
            quantite_remisee_maximale=("quantite_remise", "max"))
        resume["lignes_en_depassement"] = (meme_famille[meme_famille.quantite_remise > meme_famille.quantite_vente]
                                           .groupby("annee").size())
        resume["lignes_en_depassement"] = resume.lignes_en_depassement.fillna(0).astype(int)
        print(f"\nSous-catégorie {sous_cat.strip()}, lignes en promotion, par année :")
        display(resume)

    # Détail sur la sous-catégorie la plus touchée, l'année la plus touchée : quantité vendue × quantité remisée
    sous_cat = anomalie.sous_categorie.value_counts().index[0]
    annee_touchee = anomalie[anomalie.sous_categorie == sous_cat].annee.value_counts().index[0]
    petites = ventes[(ventes.sous_categorie == sous_cat) & (ventes.id_promotion != 1)
                     & (ventes.annee == annee_touchee) & (ventes.quantite_vente <= 4)]
    print(f"\n{sous_cat.strip()}, {annee_touchee}, lignes en promotion : quantité vendue (lignes) × quantité remisée (colonnes)")
    display(pd.crosstab(petites.quantite_vente.rename("quantité vendue"),
                        petites.quantite_remise.rename("quantité remisée")))

**Ce que montrent les tests**

- **Le coût et le chiffre d'affaires portent sur la quantité vendue**, la remise sur la quantité remisée. Pour 2 unités vendues et 4 remisées à 20 %, la ligne enregistre le coût de 2 unités, le prix de 2 unités, et une remise de 4 × 20 % du prix.
- **H1 (unités offertes) est écartée** : la remise vaut toujours le taux annoncé appliqué à la quantité remisée, jamais le prix plein d'unités gratuites, et le coût ne porte pas sur plus d'unités que la vente.
- **H3 (pack) est écartée** : la quantité remisée n'est pas un multiple de la quantité vendue (on trouve 3 pour 2, 4 pour 3, 7 pour 6), le dépassement existe à tous les taux, et dans les autres boutiques la même promotion sur le même produit remise 0, 1 ou toutes les unités, sans jamais dépasser la quantité vendue.
- **H2 (erreur sur la quantité remisée) est la seule cohérente.** La quantité remisée prend les valeurs 0 à 4 avec la même fréquence, quelle que soit la quantité vendue. Le dépassement n'apparaît donc que les années où ces produits se vendent par 2 ou 3 : *Download Games* en 2019 (minimum 2, puis 8 en 2020 et 2021), *Washers & Dryers* en 2019 et 2021 (minimum 2, contre 4 en 2020). Quand le minimum est de 4 ou plus, la même répartition ne peut pas dépasser. C'est un effet de la génération du fichier, pas un mécanisme commercial.

**Décision de l'équipe** : on applique une règle de bon sens, vraie dans tout commerce et respectée par toutes les autres lignes : **on ne peut pas remiser plus d'unités qu'on n'en vend**. Dans `04_nettoyage` (règle N6), la quantité remisée de ces lignes est ramenée à la quantité vendue. **Les montants ne changent pas** : la remise enregistrée est conservée et s'applique aux unités vendues, et le montant des ventes reste celui de la source. Un indicateur `remise_corrigee` marque les lignes et les quantités d'origine sont conservées dans le journal de nettoyage.

**Pourquoi ne pas recalculer les montants** : le chiffre d'affaires et le coût de ces lignes sont cohérents avec la quantité vendue, et nos totaux sont égaux à ceux du plan stratégique de Contoso au cent près (`00_inventaire`, section 9). Recalculer la remise aurait augmenté le chiffre d'affaires de la « remise en trop » et créé un écart avec les charges du plan.

**Limite assumée** : sur ces lignes, la remise conservée dépasse ce que donnerait le taux annoncé appliqué aux unités vendues. Leur taux de remise effectif est donc supérieur au taux annoncé (jusqu'à 40 % pour une promotion de 20 %). L'indicateur `remise_corrigee` permet de les isoler dans les analyses de remise. L'enjeu est de 0,33 % des lignes, 0,09 % du chiffre d'affaires et 0,57 % du montant des remises (292 112 $ de remise au-delà des unités vendues), dont plus des quatre cinquièmes en 2019.

### R4 : le coût total, un cas à part

Le coût total sert à calculer la **marge en valeur**, qui est notre mesure centrale. On le contrôle donc plus en détail : on compare la règle simple (coût unitaire × quantité vendue) et la variante où les unités retournées sont retirées, en séparant les lignes avec et sans retour.

In [ ]:
# Deux formules candidates pour le coût total
r4_brut = ecart(ventes.cout_total, cu * q)            # coût de toutes les unités vendues
r4_net = ecart(ventes.cout_total, cu * (q - qret))    # coût des unités vendues et non retournées

avec_retour = qret > 0
tableau_r4 = pd.DataFrame({
    "lignes": [(~avec_retour).sum(), avec_retour.sum()],
    "écart avec coût × quantité": [(r4_brut & ~avec_retour).sum(), (r4_brut & avec_retour).sum()],
    "écart avec coût × (quantité − retour)": [(r4_net & ~avec_retour).sum(), (r4_net & avec_retour).sum()],
    "écart avec les deux formules": [(r4_brut & r4_net & ~avec_retour).sum(), (r4_brut & r4_net & avec_retour).sum()],
}, index=["sans retour", "avec retour"])
display(tableau_r4)

noter("7. Règles", "R4 non respectée : coût total ≠ coût unitaire × quantité vendue", r4_brut.sum(),
      "À décider", "Laisser cout_total intact ; recalculer cout_brut, cout_retour et cout_net avec une formule unique dans 05_variables")

# ---------------------------------------------------------------------------
# Comprendre l'écart : combien d'unités le coût total compte-t-il réellement ?
# ---------------------------------------------------------------------------
# On ne voit pas la formule utilisée par la source, mais on peut la retrouver :
#     coût total / coût unitaire = nombre d'unités comptées dans le coût
# Exemple réel : 16 unités vendues, 1 retournée, coût unitaire 224,87 $.
#     16 × 224,87 = 3 597,92 $   mais le fichier indique 3 373,05 $ = 15 × 224,87
#     -> le coût de l'unité retournée a été enlevé.
# On compare ensuite les unités enlevées du coût à la quantité retournée déclarée.
unites_comptees = (ventes.cout_total / cu).round(2)
unites_enlevees = q - unites_comptees
logique = np.select(
    [unites_enlevees == 0, unites_enlevees == qret, unites_enlevees > qret],
    ["coût de toutes les unités vendues (retour ignoré)",
     "coût des unités vendues moins les unités retournées",
     "coût réduit de PLUS d'unités que le retour déclaré"],
    default="autre")
logique = pd.Series(logique, index=ventes.index)

# Sans retour, une seule logique possible (rien à enlever). On regarde donc les lignes avec retour.
print("\nLogique du coût total sur les lignes AVEC retour, par année :")
logique_par_annee = pd.crosstab(logique[avec_retour], ventes.annee[avec_retour], margins=True, margins_name="total")
display(logique_par_annee)

# La logique dépend-elle de la taille de la ligne ? (dernière année, où deux logiques coexistent)
tranches_q = pd.cut(q, [0, 7, 12, 20, 50, 5000], labels=["6", "8 à 12", "13 à 20", "21 à 50", "plus de 50"])
derniere_annee = avec_retour & (ventes.annee == ventes.annee.max())
print(f"Année {ventes.annee.max()} : logique du coût total selon la quantité vendue (part des lignes avec retour) :")
display(pd.crosstab(logique[derniere_annee], tranches_q[derniere_annee], normalize="columns").style.format("{:.0%}"))

# Ampleur de l'écart : en dollars et en part du coût total
ecart_dollars = (cu * q - ventes.cout_total)[r4_brut]
print(f"\nÉcart cumulé sur ces lignes : {ecart_dollars.sum():,.0f} $".replace(",", " "),
      f"soit {ecart_dollars.sum() / ventes.cout_total.sum():.2%} du coût total")
print("\nExemples de lignes en écart :")
display(ventes.loc[r4_brut, ["id_vente", "quantite_vente", "quantite_retour", "cout_unitaire", "cout_total"]]
        .assign(cout_x_quantite=lambda d: d.cout_unitaire * d.quantite_vente,
                cout_x_quantite_nette=lambda d: d.cout_unitaire * (d.quantite_vente - d.quantite_retour))
        .head(8))

# Lecture : la source calcule le coût total de trois façons selon les lignes. La répartition change
# brutalement d'une année à l'autre (moitié-moitié en 2019, presque tout en coût net en 2020, de nouveau
# partagée en 2021) et dépend de la quantité vendue : ce n'est pas du hasard, c'est une
# règle de calcul qui n'est pas la même partout. On ne peut pas savoir pourquoi.
# Conséquence : une marge calculée avec cette colonne mélange coût brut et coût net des retours.
# Décision : on ne modifie pas cout_total ; 05_variables recalcule trois colonnes homogènes
#     cout_brut   = cout_unitaire × quantite_vente
#     cout_retour = cout_unitaire × quantite_retour
#     cout_net    = cout_brut − cout_retour

# Marge en valeur calculée avec le coût total tel qu'il est enregistré
marge = ventes.montant_ventes - ventes.cout_total
noter("7. Règles", "Marge en valeur négative (montant des ventes < coût total)", (marge < 0).sum(),
      "Information", "Garder : une vente à perte est une information utile pour Q5")

## 8. Cohérence des promotions

Deux questions : la promotion était-elle **en cours** le jour de la vente, et une vente rattachée à une promotion porte-t-elle **réellement** une remise ? La réponse conditionne Q2 (effet des promotions) et Q4 (lien remise et retours).

In [ ]:
# ---------------------------------------------------------------------------
# 1. Vente datée en dehors de la période de sa promotion
# ---------------------------------------------------------------------------
hors_periode = (ventes.date < ventes.promo_date_debut) | (ventes.date > ventes.promo_date_fin)
detail_hors = (ventes[hors_periode]
               .groupby(["id_promotion", "nom_promotion", "promo_date_debut", "promo_date_fin"])
               .agg(nb_ventes=("id_vente", "size"), premiere_vente=("date", "min"), derniere_vente=("date", "max"))
               .reset_index())
display(detail_hors)
noter("8. Promotions", "Ventes datées hors de la période de leur promotion", hors_periode.sum(),
      "À décider", "Garder la vente et sa remise (elles sont réelles) ; ajouter un indicateur promo_hors_periode")

# ---------------------------------------------------------------------------
# 1 bis. Comprendre l'anomalie : existe-t-il une promotion "jumelle" valide à ces dates ?
# ---------------------------------------------------------------------------
# Démarche : pour chaque promotion en anomalie, on cherche dans dim_promotion une autre
# promotion qui porte le MÊME NOM et le MÊME TAUX, et dont la période contient bien les
# dates des ventes concernées. Si elle existe, l'explication la plus simple est une erreur
# d'identifiant dans la source (la bonne édition de la promotion n'a pas été choisie).
dim_promotion = pd.read_parquet(config.DIM_REALIGNEES / "dim_promotion.parquet")
jumelles = []
for _, anomalie in detail_hors.iterrows():
    candidates = dim_promotion[(dim_promotion.nom_promotion == anomalie.nom_promotion)
                               & (dim_promotion.taux_remise_annonce == dim_promotion.loc[
                                   dim_promotion.id_promotion == anomalie.id_promotion, "taux_remise_annonce"].iloc[0])
                               & (dim_promotion.date_debut <= anomalie.premiere_vente)
                               & (dim_promotion.date_fin >= anomalie.derniere_vente)]
    for _, c in candidates.iterrows():
        jumelles.append({"promotion_en_anomalie": anomalie.id_promotion, "promotion_jumelle": c.id_promotion,
                         "nom": c.nom_promotion, "taux": c.taux_remise_annonce,
                         "debut_jumelle": c.date_debut, "fin_jumelle": c.date_fin,
                         "ventes_de_la_jumelle_sur_la_meme_periode": int((
                             (ventes.id_promotion == c.id_promotion)
                             & ventes.date.between(anomalie.premiere_vente, anomalie.derniere_vente)).sum())})
jumelles = pd.DataFrame(jumelles)
print("Promotions jumelles (même nom, même taux, période valide pour les ventes en anomalie) :")
display(jumelles)
# Lecture : si la jumelle existe et porte elle-même des ventes sur la même période, les deux
# identifiants décrivent la même opération commerciale. La remise est réelle, seul l'identifiant
# d'édition est faux. Pour l'analyse des promotions, on regroupera par NOM de promotion.

# ---------------------------------------------------------------------------
# 2. Promotion et remise : les quatre combinaisons possibles
# ---------------------------------------------------------------------------
# id_promotion = 1 correspond à "No Discount" : la vente n'est rattachée à aucune promotion.
a_promo = ventes.id_promotion != 1
a_remise = ventes.montant_remise > 0
croisement = pd.crosstab(a_promo.map({True: "avec promotion", False: "sans promotion"}),
                         a_remise.map({True: "remise > 0", False: "remise = 0"}))
display(croisement)

noter("8. Promotions", "Ventes sans promotion mais avec une remise", (~a_promo & a_remise).sum(),
      "À décider", "Vérifier l'identifiant de promotion")
noter("8. Promotions", "Ventes rattachées à une promotion mais sans aucune unité remisée", (a_promo & ~a_remise).sum(),
      "Information", "Créer la variable a_remise (montant_remise > 0), distincte de l'appartenance à une promotion")

# ---------------------------------------------------------------------------
# 3. Quelle part des unités d'une vente en promotion est réellement remisée ?
# ---------------------------------------------------------------------------
part_remisee = (qr / q)[a_promo]
print("\nPart des unités remisées, parmi les ventes rattachées à une promotion :")
display(part_remisee.describe(percentiles=[.25, .5, .75, .95]).to_frame("part_unites_remisees").T)

# ---------------------------------------------------------------------------
# 4. Quel mécanisme de remise ? Test de trois hypothèses
# ---------------------------------------------------------------------------
# Point de départ : id_promotion fait partie de la clé de la ligne. Les unités vendues SANS
# promotion sont donc sur une autre ligne (id_promotion = 1). Une ligne rattachée à une
# promotion ne mélange pas "clients avec promo" et "clients sans promo" : c'est bien à
# l'intérieur d'une même promotion que seule une partie des unités est remisée.
#
# Hypothèses sur le mécanisme, et ce que chacune prédit pour la part d'unités remisées :
#   H1 "remise sur tout l'achat"        -> part = 100 %
#   H2 "remise sur la 2e unité"         -> part = 50 %  (taux effectif = moitié du taux annoncé)
#   H3 "remise sur la 3e unité"         -> part = 33 %
# On compare ces prédictions à la part observée (à 2 points près).
remisees = part_remisee[part_remisee > 0]
hypotheses = pd.DataFrame({
    "part_predite": [1.0, 0.5, 1 / 3],
    "lignes_compatibles": [((remisees - cible).abs() <= 0.02).sum() for cible in (1.0, 0.5, 1 / 3)],
}, index=["H1 remise sur tout l'achat", "H2 remise sur la 2e unité", "H3 remise sur la 3e unité"])
hypotheses["part_des_lignes_remisees"] = hypotheses.lignes_compatibles / len(remisees)
display(hypotheses.style.format({"part_predite": "{:.0%}", "lignes_compatibles": "{:,.0f}", "part_des_lignes_remisees": "{:.1%}"}))

# Taux de remise effectif = montant de la remise / montant avant remise, comparé au taux annoncé
taux_effectif = (ventes.montant_remise / (pu * q))[a_promo]
comparaison_taux = (pd.DataFrame({"taux_annonce": ventes.taux_remise_annonce[a_promo], "taux_effectif": taux_effectif})
                    .groupby("taux_annonce").taux_effectif.agg(lignes="size", moyenne="mean", mediane="median", maximum="max"))
print("Taux de remise effectif selon le taux annoncé :")
display(comparaison_taux.style.format({"lignes": "{:,.0f}", "moyenne": "{:.2%}", "mediane": "{:.2%}", "maximum": "{:.2%}"}))

# Écart entre le taux obtenu et le taux annoncé : combien de lignes en promotion obtiennent
# réellement le taux annoncé (à 0,1 point près) ?
au_taux_annonce = (taux_effectif - ventes.taux_remise_annonce[a_promo]).abs() <= 0.001
noter("8. Promotions", "Ventes en promotion dont le taux de remise effectif est inférieur au taux annoncé",
      (~au_taux_annonce).sum(), "Information",
      "Incohérence à signaler ; utiliser le taux effectif (calculé dans 05_variables) pour Q2 et Q4", total=int(a_promo.sum()))

**Lecture : les ventes hors période sont celles de janvier.** La promotion *European Holiday* court d'octobre à la fin janvier de l'année suivante. Les ventes en anomalie sont toutes datées de janvier et rattachées à l'édition qui ne commence qu'en octobre de la même année :

- **janvier 2021** (1 258 lignes) : l'édition valide à ces dates existe (promotion 19, octobre 2020 à janvier 2021) et porte elle-même 11 496 ventes sur le même mois. Les deux identifiants décrivent la même opération : seul l'identifiant d'édition est faux ;
- **janvier 2019** (28 454 lignes) : aucune promotion jumelle n'est trouvée, parce que l'édition précédente (octobre 2018 à janvier 2019) n'existe pas dans la table des promotions, qui commence en 2019. Toutes les ventes de fin de campagne de janvier 2019 ont donc été rattachées à l'édition 2019. C'est la même situation, sans jumelle pour le prouver.

En janvier 2020, l'édition 2019 est dans sa période : aucune anomalie. **Conséquence** : la remise de ces lignes est réelle et conservée. On marque les lignes avec `promo_hors_periode` et, pour l'analyse, on regroupe les promotions par nom et non par identifiant d'édition. Avec 2019, l'anomalie passe de 1 258 à 29 712 lignes (4,15 %) : ce n'est plus un détail, d'où l'importance de la traiter par un indicateur et non par une exclusion.

### La table des promotions explique-t-elle le mécanisme ?

Avant de conclure, on retourne à la source : la table brute `DimPromotion` de Contoso. Dans `02_flocon`, on n'en a gardé que les colonnes utiles. On regarde ici **toutes** les colonnes qui pourraient décrire une règle de remise :

- `PromotionDescription` : un texte qui pourrait expliquer la promotion ;
- `MinQuantity` et `MaxQuantity` : la quantité minimale et maximale donnant droit à la remise. C'est exactement là qu'on trouverait une règle du type « à partir de 2 unités » ou « limitée à 4 unités ».

In [ ]:
# ---------------------------------------------------------------------------
# 5. Retour à la table brute DimPromotion (data/raw/contoso/Dim)
# ---------------------------------------------------------------------------
promotions_brutes, _ = etl.lire_dimension("DimPromotion")
print("Colonnes de DimPromotion :", promotions_brutes.columns.tolist())
display(promotions_brutes[["PromotionKey", "PromotionName", "PromotionDescription",
                           "DiscountPercent", "PromotionType", "PromotionCategory", "MinQuantity", "MaxQuantity"]])

# a) La description apporte-t-elle une information de plus que le nom ?
description_differente = (promotions_brutes.PromotionDescription.astype(str).str.strip()
                          != promotions_brutes.PromotionName.astype(str).str.strip()).sum()
print("Promotions dont la description diffère du nom :", int(description_differente), "sur", len(promotions_brutes))

# b) Les quantités minimale et maximale sont-elles renseignées ?
min_renseignees = int(promotions_brutes.MinQuantity.notna().sum())
max_renseignees = int(promotions_brutes.MaxQuantity.notna().sum())
print("Promotions avec MinQuantity renseignée :", min_renseignees, "sur", len(promotions_brutes))
print("Promotions avec MaxQuantity renseignée :", max_renseignees, "sur", len(promotions_brutes))

# c) Le type et la catégorie distinguent-ils plusieurs sortes de promotions ?
vraies_promotions = promotions_brutes[promotions_brutes.PromotionName != "No Discount"]
print("Types de promotion (hors 'No Discount') :", vraies_promotions.PromotionType.unique().tolist())
print("Catégories de promotion (hors 'No Discount') :", vraies_promotions.PromotionCategory.unique().tolist())

# Nombre de promotions pour lesquelles AUCUNE colonne ne décrit la règle de remise
sans_regle = ((promotions_brutes.PromotionDescription.astype(str).str.strip() == promotions_brutes.PromotionName.astype(str).str.strip())
              & promotions_brutes.MinQuantity.isna() & promotions_brutes.MaxQuantity.isna()).sum()
noter("8. Promotions", "Promotions sans aucune règle décrite (description = nom, quantités min et max vides)",
      sans_regle, "Information", "Limite à écrire dans le rapport : mécanisme des promotions inconnu",
      total=len(promotions_brutes))

**Lecture** : être rattachée à une promotion ne veut pas dire que toute la vente est remisée. La remise ne porte que sur une partie des unités, et parfois sur aucune. 

**Conclusion sur les promotions**

- **Ce qui est sûr** : les remises ne correspondent pas au taux annoncé. Le taux réellement obtenu est presque toujours inférieur, parce que seule une partie des unités est remisée. Les rares lignes où il le dépasse sont celles de la règle R5 (remise sur plus d'unités que la vente, section 7).
- **Ce qu'on ne peut pas savoir** : pourquoi. La part d'unités remisées ne suit aucune règle fixe (ni 2e unité, ni 3e unité), et la table des promotions ne décrit aucun mécanisme : la description recopie le nom, les quantités minimale et maximale sont vides, et toutes les promotions ont le même type et la même catégorie.
- **Nous n'avons donc aucun moyen de comprendre le mécanisme des promotions.** C'est une incohérence des données, signalée comme telle. Pour mesurer l'effet des remises, on utilisera donc le **taux de remise effectif** (montant de la remise ÷ montant avant remise), calculé dans `05_variables`, et non le seul taux annoncé.

## 9. Cohérence dans le temps

Les ventes complètes couvrent les douze trimestres de 2019 à 2021. On vérifie qu'aucun jour ne manque, puis on regarde l'activité de chaque boutique : depuis quand vend-elle, et y a-t-il des mois sans vente ?

*Changement par rapport aux fichiers du professeur : les 4es trimestres y étaient absents, et 2019 aussi. Avec trois années entières, on voit maintenant les ouvertures de magasins, qui étaient invisibles sur deux ans.*

In [ ]:
# ---------------------------------------------------------------------------
# 1. Jours sans aucune vente, sur la période attendue
# ---------------------------------------------------------------------------
tous_les_jours = pd.date_range(config.DATE_DEBUT, config.DATE_FIN, freq="D")
jours_vendus = pd.DatetimeIndex(ventes.date.dt.normalize().unique())
jours_absents = tous_les_jours.difference(jours_vendus)

# Avec les ventes complètes, chaque jour des trois années doit porter au moins une vente.
# On liste les trimestres touchés s'il y en a (avec les fichiers du professeur, ce seraient les T4).
trimestres_touches = sorted({f"{j.year} T{j.quarter}" for j in jours_absents})
print("Jours de la période :", len(tous_les_jours), "· jours sans aucune vente :", len(jours_absents),
      "· trimestres touchés :", trimestres_touches or "aucun")
noter("9. Temps", "Jours sans aucune vente sur la période", len(jours_absents),
      "À décider", "Vérifier le fichier du trimestre concerné", total=len(tous_les_jours))

# ---------------------------------------------------------------------------
# 2. Ventes hors de la période attendue, ou avant l'ouverture de la boutique
# ---------------------------------------------------------------------------
hors_periode_projet = ~ventes.date.between(pd.Timestamp(config.DATE_DEBUT), pd.Timestamp(config.DATE_FIN))
noter("9. Temps", "Ventes datées hors de la période du projet", hors_periode_projet.sum(), "À décider", "Exclure")
avant_ouverture = ventes.date < ventes.date_ouverture
noter("9. Temps", "Ventes datées avant l'ouverture de la boutique", avant_ouverture.sum(),
      "À décider", "Garder (elles sont dans les totaux du plan) ; ajouter l'indicateur avant_ouverture ; "
                   "comparer les magasins sur leurs mois d'activité après ouverture")

# ---------------------------------------------------------------------------
# 3. Activité de chaque boutique : première vente, dernière vente, mois sans vente
# ---------------------------------------------------------------------------
mois_de_vente = ventes.date.dt.to_period("M")
nb_mois_periode = len(pd.period_range(config.DATE_DEBUT, config.DATE_FIN, freq="M"))
activite = (ventes.assign(mois_de_vente=mois_de_vente).groupby(["nom_boutique", "pays"])
            .agg(premiere_vente=("date", "min"), derniere_vente=("date", "max"),
                 jours_actifs=("date", "nunique"), mois_actifs=("mois_de_vente", "nunique"),
                 nb_ventes=("id_vente", "size"))
            .reset_index())
activite["mois_sans_vente"] = nb_mois_periode - activite.mois_actifs

# On ajoute la date d'ouverture et la date de fermeture venant de dim_boutique, pour EXPLIQUER
# une activité partielle : boutique ouverte en cours de période, ou boutique fermée.
dates_boutiques = ventes.groupby("nom_boutique")[["date_ouverture", "date_fermeture"]].first().reset_index()
activite = activite.merge(dates_boutiques, on="nom_boutique", how="left")

# Ventes enregistrées AVANT la date d'ouverture : nombre de lignes, montant, et ce que la boutique
# vend ensuite (pour comparer le rythme avant et après l'ouverture).
avant = (ventes[avant_ouverture].groupby("nom_boutique")
         .agg(lignes_avant_ouverture=("id_vente", "size"), ca_avant_ouverture=("montant_ventes", "sum"),
              derniere_vente_avant_ouverture=("date", "max")))
apres = (ventes[~avant_ouverture].groupby("nom_boutique")
         .agg(lignes_apres_ouverture=("id_vente", "size"), premiere_vente_apres_ouverture=("date", "min")))
activite = activite.merge(avant, on="nom_boutique", how="left").merge(apres, on="nom_boutique", how="left")
activite["lignes_avant_ouverture"] = activite.lignes_avant_ouverture.fillna(0).astype(int)
# Nombre de jours entre l'ouverture et la première vente qui la suit (0 = la boutique vend dès l'ouverture)
activite["jours_ouverture_a_premiere_vente"] = (activite.premiere_vente_apres_ouverture - activite.date_ouverture).dt.days

# Une boutique est "partielle" si elle ouvre pendant la période étudiée.
debut = pd.Timestamp(config.DATE_DEBUT)
partielles = activite[activite.date_ouverture > debut].sort_values("date_ouverture")
print("Boutiques ouvertes pendant la période étudiée :")
display(partielles[["nom_boutique", "pays", "date_ouverture", "premiere_vente", "lignes_avant_ouverture",
                    "ca_avant_ouverture", "derniere_vente_avant_ouverture", "premiere_vente_apres_ouverture",
                    "jours_ouverture_a_premiere_vente", "lignes_apres_ouverture", "mois_sans_vente"]]
        .style.format({"ca_avant_ouverture": "{:,.0f}", "date_ouverture": "{:%Y-%m-%d}", "premiere_vente": "{:%Y-%m-%d}",
                       "derniere_vente_avant_ouverture": "{:%Y-%m-%d}", "premiere_vente_apres_ouverture": "{:%Y-%m-%d}"})
        .hide(axis="index"))
ca_avant = float(ventes.loc[avant_ouverture, "montant_ventes"].sum())
print(f"Ventes avant ouverture : {int(avant_ouverture.sum())} lignes · {ca_avant:,.0f} $ ".replace(",", " ")
      + f"({ca_avant / float(ventes.montant_ventes.sum()):.2%} du chiffre d'affaires) · années : "
      + str(ventes.loc[avant_ouverture, "annee"].value_counts().to_dict()))

# Toutes les autres boutiques vendent-elles chaque mois de la période ?
autres = activite[activite.date_ouverture <= debut]
print("Autres boutiques :", len(autres), "· avec au moins un mois sans vente :", int((autres.mois_sans_vente > 0).sum()))
print("Boutiques de nos ventes avec une date de fermeture :", int(activite.date_fermeture.notna().sum()))
# Nombre de boutiques qui vendent, par année
print("Boutiques avec au moins une vente, par année :", ventes.groupby("annee").id_boutique.nunique().to_dict())
noter("9. Temps", "Boutiques ouvertes pendant la période (activité partielle)", len(partielles),
      "Information", "Comparer les boutiques sur une moyenne par mois d'activité, pas sur un total", total=len(activite))

**Lecture : six magasins ouvrent pendant les trois années.** Trois ouvrent en 2019 (Glasgow en août, Düsseldorf en septembre, Roma en octobre), deux en 2020 (Marseille en avril, Berlin en octobre) et un en 2021 (Venezia en juillet). Pour chacun, la première vente régulière tombe **le jour même de l'ouverture** : les dates d'ouverture de la table des boutiques sont donc fiables.

**L'anomalie** : ces six magasins portent quand même 189 lignes de vente datées d'avant leur ouverture, toutes en 2019, pour 1,14 M$ (0,05 % du chiffre d'affaires). Ce sont des ventes éparses, 20 à 45 lignes par magasin sur l'année, là où un magasin ouvert en enregistre plus de mille. Elles s'arrêtent avant l'ouverture, et entre les deux la boutique ne vend rien (Venezia : rien pendant toute l'année 2020 et le premier semestre 2021 ; Berlin : rien de janvier à septembre 2020 ; Marseille : rien de janvier à mars 2020). Sur deux années, on ne les voyait pas : 2020 et 2021 n'en contiennent aucune.

**Décision proposée** (règle N8 de `04_nettoyage`) : on **garde** ces lignes, parce qu'elles font partie des totaux qui recoupent le plan stratégique au cent près, et on les marque avec l'indicateur `avant_ouverture`. Les supprimer créerait un écart de 1,14 M$ avec les charges. En revanche, toute comparaison entre magasins (ventes par magasin, résultat par magasin, magasins bénéficiaires) doit se faire sur les **mois d'activité après ouverture** : un magasin ouvert en octobre ne se compare pas sur un total annuel à un magasin ouvert toute l'année. C'est le biais qui avait faussé une première estimation du regroupement des magasins.

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : nombre de ventes par semaine. Un trou ou une chute brutale se voit tout de suite.
# ---------------------------------------------------------------------------
par_semaine = ventes.set_index("date").resample("W").size()

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(par_semaine.index, par_semaine.where(par_semaine > 0), color=viz.PRINCIPALE, linewidth=1.8)

# Bande claire sur chaque mois sans aucune vente, s'il y en a (même convention que dans 00_inventaire).
# Avec les ventes complètes il n'y en a pas : la courbe est continue sur les trois années.
ventes_par_mois = ventes.set_index("date").resample("MS").size()
for debut_mois in ventes_par_mois[ventes_par_mois == 0].index:
    ax.axvspan(debut_mois, debut_mois + pd.offsets.MonthEnd(1), color=viz.CLAIR, zorder=0)
ax.set_ylim(0, par_semaine.max() * 1.1)
ax.set_ylabel("Nombre de lignes de vente")
# Le sous-titre dit ce que le graphique montre réellement : on ne parle de zones grisées que s'il y en a.
nb_mois_vides = int((ventes_par_mois == 0).sum())
constat = "zones grisées : mois sans aucune vente" if nb_mois_vides else "aucun mois sans vente"
viz.titre(ax, "Lignes de vente par semaine", f"Contrôle de continuité · toutes boutiques · {constat}")
print("Figure enregistrée :", viz.enregistrer(fig, "03_ventes_par_semaine"))
plt.show()

## 10. Valeurs extrêmes

Méthode de l'**écart interquartile** (IQR) : on calcule Q1 (25 % des valeurs sont en dessous) et Q3 (75 % sont en dessous). L'écart IQR = Q3 − Q1 mesure l'étendue de la moitié centrale. Une valeur est dite extrême si elle dépasse Q3 + 1,5 × IQR ou descend sous Q1 − 1,5 × IQR.

**Extrême ne veut pas dire faux.** Une commande de 2 000 unités par un revendeur est rare mais réelle. On cherche donc à comprendre d'où viennent ces valeurs avant de décider quoi que ce soit.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Bornes IQR et nombre de valeurs extrêmes par mesure
# ---------------------------------------------------------------------------
def bornes_iqr(serie):
    """Renvoie (borne basse, borne haute) de la méthode de l'écart interquartile."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


lignes = []
for col in ["quantite_vente", "prix_unitaire", "montant_ventes"]:
    bas, haut = bornes_iqr(ventes[col])
    extremes = (ventes[col] < bas) | (ventes[col] > haut)
    lignes.append({"mesure": col, "borne_haute": haut, "nb_extremes": extremes.sum(),
                   "part": extremes.mean(), "centile_99": ventes[col].quantile(0.99), "maximum": ventes[col].max(),
                   "part_du_total": ventes.loc[extremes, col].sum() / ventes[col].sum()})
extremes_tab = pd.DataFrame(lignes).set_index("mesure")
display(extremes_tab.style.format({"part": "{:.2%}", "part_du_total": "{:.1%}", "borne_haute": "{:,.1f}",
                                   "centile_99": "{:,.1f}", "maximum": "{:,.1f}", "nb_extremes": "{:,.0f}"}))

# ---------------------------------------------------------------------------
# 2. D'où viennent les grandes quantités ? Répartition par canal
# ---------------------------------------------------------------------------
bas, haut = bornes_iqr(ventes.quantite_vente)
grandes = ventes.quantite_vente > haut
display(ventes.groupby("canal").agg(nb_ventes=("id_vente", "size"),
                                    quantite_mediane=("quantite_vente", "median"),
                                    quantite_max=("quantite_vente", "max"),
                                    part_grandes_quantites=("quantite_vente", lambda s: (s > haut).mean())))

print("Les cinq plus grandes quantités :")
display(ventes.nlargest(5, "quantite_vente")[["date", "nom_boutique", "canal", "nom_produit",
                                              "quantite_vente", "prix_unitaire", "montant_ventes"]])

# Ces lignes respectent-elles la règle R1 ? Si oui, ce sont de grosses ventes cohérentes, pas des erreurs de saisie.
print("Grandes quantités qui ne respectent pas R1 :", int((grandes & r1).sum()), "sur", int(grandes.sum()))

noter("10. Extrêmes", "Quantités vendues au-dessus de la borne IQR", grandes.sum(), "À décider",
      "Garder (ventes cohérentes) ; ajouter un indicateur grande_quantite et utiliser la médiane ou une échelle logarithmique")
bas_m, haut_m = bornes_iqr(ventes.montant_ventes)
noter("10. Extrêmes", "Montants des ventes au-dessus de la borne IQR", (ventes.montant_ventes > haut_m).sum(),
      "À décider", "Garder ; même traitement que les grandes quantités")

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : distribution de la quantité vendue par type de boutique (échelle logarithmique)
# ---------------------------------------------------------------------------
# L'échelle logarithmique est nécessaire : la plupart des lignes sont entre 6 et 20 unités,
# quelques-unes dépassent 1 000. Sur une échelle normale, les boîtes seraient écrasées.
# Dans nos données, le canal de vente et le type de boutique sont la même information :
# chaque boutique vend par un seul canal (Online = la boutique en ligne, Reseller = le revendeur,
# Store = les magasins). Les graphiques utilisent les libellés français, comme les notebooks d'analyse.
LIBELLES_TYPES = {"Online": "En ligne", "Reseller": "Revendeur", "Store": "Magasins"}
canaux = [c for c in LIBELLES_TYPES if c in set(ventes.canal)]
fig, ax = plt.subplots(figsize=(9, 4.5))
boites = ax.boxplot([ventes.loc[ventes.canal == c, "quantite_vente"] for c in canaux],
                    patch_artist=True, widths=0.55,
                    flierprops=dict(marker="o", markersize=3, markerfacecolor=viz.ALERTE,
                                    markeredgecolor="none", alpha=0.35),
                    medianprops=dict(color=viz.ACCENT, linewidth=2))
for boite in boites["boxes"]:
    boite.set(facecolor=viz.CLAIR, edgecolor=viz.PRINCIPALE)
ax.set_xticks(range(1, len(canaux) + 1), [LIBELLES_TYPES[c] for c in canaux])
ax.set_yscale("log")
ax.set_ylabel("Quantité vendue par ligne (échelle log)")
viz.titre(ax, "Quantité vendue par ligne, selon le type de boutique",
          "Trait orange : médiane · points rouges : valeurs au-delà de 1,5 × IQR")
print("Figure enregistrée :", viz.enregistrer(fig, "03_quantites_par_canal"))
plt.show()

## 11. Nature des données : un jeu généré, pas une activité réelle

Les sections précédentes montrent des données très propres. Cette section pose une autre question : **ces ventes ressemblent-elles à celles d'un vrai commerce ?** On teste cinq propriétés qu'une vraie activité aurait forcément, et on regarde si les données les respectent.

| Test | Dans un vrai commerce | Ce qu'on vérifie |
|---|---|---|
| A. Grain de la ligne | Une ligne = un ticket, rattaché à un client | Colonnes disponibles, unicité de la clé métier |
| B. Quantités | Beaucoup de ventes à 1 unité, surtout pour les produits chers | Quantité minimale, valeurs possibles |
| C. Prix | Les prix bougent (inflation, baisse des produits électroniques) | Nombre de prix différents par produit |
| D. Régularité | L'activité varie selon le jour de la semaine | Lignes par jour de semaine |
| E. COVID-19 | Chute des ventes en magasin pendant les confinements, report vers la vente en ligne | Ventes par trimestre et par canal face à l'indice de sévérité, mois d'avant les restrictions |

Rappel : l'inventaire (`00_inventaire`) a établi que ces fichiers sont un extrait de la base de démonstration Contoso de Microsoft, dont les dates ont été décalées de 12 ans (2007-2009 → 2019-2021).

In [ ]:
# ---------------------------------------------------------------------------
# A. Grain de la ligne : ticket de caisse ou cumul ?
# ---------------------------------------------------------------------------
# id_vente existe et il est unique, mais c'est un numéro de LIGNE, pas un numéro de ticket :
# la section 4 a montré qu'il n'y a jamais deux lignes pour le même jour, la même boutique,
# le même produit, la même promotion et le même canal. Une ligne est donc un cumul.
colonnes_client = [c for c in ventes.columns if "client" in c.lower() or "customer" in c.lower() or "ticket" in c.lower()]
print("A. Colonnes identifiant un client ou un ticket :", colonnes_client or "aucune")
print("   Conséquence : impossible de calculer un panier moyen, un nombre de clients par jour,")
print("   une fréquence d'achat ou une fidélité. Ces analyses sont hors de portée du jeu de données.")

# Vérification du grain : combien de lignes par combinaison (jour, boutique, produit, promotion, canal) ?
lignes_par_cle = ventes.groupby(CLE_METIER).size()
print("   Nombre maximal de lignes pour une même combinaison jour × boutique × produit × promotion × canal :",
      int(lignes_par_cle.max()))
# Dans un magasin réel, un même produit est vendu à plusieurs clients le même jour : on aurait
# plusieurs tickets, donc plusieurs lignes par combinaison. Ici il n'y en a jamais qu'une.
# Cette table n'est donc pas le journal des ventes (une ligne par passage en caisse) : c'est une
# table CONSOLIDÉE, comme celles qu'on produit après la clôture de la journée en additionnant
# les tickets. Le détail d'origine (tickets, clients, heures) n'est pas fourni.
noter("11. Nature", "Colonnes identifiant un client ou un ticket de caisse (0 = analyses client impossibles)",
      len(colonnes_client), "Information", "Écrire cette limite dans le rapport", total=ventes.shape[1], zero_est_ok=False)

# ---------------------------------------------------------------------------
# B. Quantités : aucune vente à l'unité, même pour les produits les plus chers
# ---------------------------------------------------------------------------
print("\nB. Quantité minimale par ligne :", ventes.quantite_vente.min())
print("   Lignes d'une seule unité :", int((ventes.quantite_vente == 1).sum()))
moins_de_6 = ventes[ventes.quantite_vente < 6]
print("   Lignes avec moins de 6 unités :", len(moins_de_6), f"({len(moins_de_6) / len(ventes):.1%})",
      "· catégories :", moins_de_6.categorie.str.strip().value_counts().to_dict())
# Quantité minimale par catégorie et par année
display(ventes.groupby([ventes.categorie.str.strip(), "annee"]).quantite_vente.min().unstack("annee")
        .assign(lignes=ventes.groupby(ventes.categorie.str.strip()).size()))
print("   Part des lignes de moins de 6 unités, par année :",
      {a: f"{p:.1%}" for a, p in (ventes.quantite_vente < 6).groupby(ventes.annee).mean().items()})
print("   Nombre de valeurs différentes prises par la quantité :", ventes.quantite_vente.nunique(),
      "sur", f"{len(ventes):,}".replace(",", " "), "lignes")

# Les 8 quantités les plus fréquentes et la part des lignes qu'elles couvrent
frequentes = ventes.quantite_vente.value_counts().head(8)
print("   Les 8 quantités les plus fréquentes :", frequentes.index.tolist(),
      f"-> {frequentes.sum() / len(ventes):.1%} des lignes")

# Quantité minimale par tranche de prix : un vrai magasin vend ses téléviseurs à l'unité.
tranches = pd.cut(ventes.prix_unitaire, [0, 50, 200, 500, 1000, 5000],
                  labels=["moins de 50 $", "50 à 200 $", "200 à 500 $", "500 à 1 000 $", "plus de 1 000 $"])
display(ventes.groupby(tranches, observed=True).quantite_vente.agg(lignes="size", minimum="min", mediane="median", maximum="max"))
noter("11. Nature", "Lignes de vente d'une seule unité (0 = aucune vente à l'unité, même à plus de 1 000 $)",
      (ventes.quantite_vente == 1).sum(), "Information", "Aucune : interpréter une ligne comme un volume, pas comme un achat", zero_est_ok=False)

In [ ]:
# ---------------------------------------------------------------------------
# C. Prix : aucun mouvement de prix en trois ans
# ---------------------------------------------------------------------------
# Pour chaque produit, on compte le nombre de prix unitaires et de coûts unitaires différents.
prix_par_produit = ventes.groupby("id_produit").agg(nb_prix=("prix_unitaire", "nunique"),
                                                   nb_couts=("cout_unitaire", "nunique"),
                                                   nb_annees=("annee", "nunique"))
vendus_deux_ans = prix_par_produit[prix_par_produit.nb_annees >= 2]
print("C. Produits vendus sur au moins deux années :", len(vendus_deux_ans), "sur", len(prix_par_produit),
      "· dont vendus les trois années :", int((prix_par_produit.nb_annees == 3).sum()))
print("   Parmi eux, produits dont le prix a changé :", int((vendus_deux_ans.nb_prix > 1).sum()))
print("   Parmi eux, produits dont le coût a changé :", int((vendus_deux_ans.nb_couts > 1).sum()))
print("   Dans la réalité, les prix auraient bougé : inflation générale, et surtout baisse rapide")
print("   du prix des produits électroniques (la plupart des 8 catégories vendues ici).")
noter("11. Nature", "Produits dont le prix unitaire a changé au moins une fois en trois ans",
      (prix_par_produit.nb_prix > 1).sum(), "Information",
      "Q5 : on ne peut pas mesurer l'effet d'un changement de prix, seulement celui des remises",
      total=len(prix_par_produit), zero_est_ok=False)

# ---------------------------------------------------------------------------
# D. Régularité : l'activité dépend-elle du jour de la semaine ?
# ---------------------------------------------------------------------------
jours = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
par_jour = (ventes.groupby(ventes.date.dt.dayofweek)
            .agg(lignes=("id_vente", "size"), ca=("montant_ventes", "sum")))
par_jour.index = jours
par_jour["part_des_lignes"] = par_jour.lignes / par_jour.lignes.sum()
print("\nD. Répartition par jour de la semaine (une répartition uniforme donnerait 14,3 % par jour) :")
display(par_jour.style.format({"lignes": "{:,.0f}", "ca": "{:,.0f}", "part_des_lignes": "{:.1%}"}))
ecart_jours = par_jour.part_des_lignes.max() - par_jour.part_des_lignes.min()
print(f"   Écart entre le jour le plus fort et le plus faible : {ecart_jours * 100:.1f} point(s) de pourcentage.")

In [ ]:
# ---------------------------------------------------------------------------
# E. COVID-19 : les ventes réagissent-elles à la sévérité des restrictions ?
# ---------------------------------------------------------------------------
# Par trimestre : indice moyen de sévérité (0 = aucune restriction, 100 = confinement strict),
# chiffre d'affaires total, et chiffre d'affaires du canal boutique physique ("Store"),
# qui est celui qu'un confinement aurait dû faire chuter.
periode = ventes.date.dt.to_period("Q").rename("trimestre")
covid = ventes.groupby(periode).agg(indice_covid_moyen=("severite_covid", "mean"),
                                    lignes=("id_vente", "size"), ca_total=("montant_ventes", "sum"))
covid["ca_boutiques_physiques"] = ventes[ventes.canal == "Store"].groupby(periode).montant_ventes.sum()
display(covid.style.format({"indice_covid_moyen": "{:.1f}", "lignes": "{:,.0f}",
                            "ca_total": "{:,.0f}", "ca_boutiques_physiques": "{:,.0f}"}))

# Corrélation au niveau le plus fin disponible : pays × jour, sur le canal boutique physique.
# On la calcule sur 2020 et 2021 seulement : en 2019 l'indice vaut 0 partout (pas de pandémie), et
# comme les magasins vendent davantage en 2019, garder cette année créerait une corrélation négative
# qui ne dirait rien des restrictions. On teste donc : à l'intérieur des années COVID, un pays vend-il
# moins les jours où ses restrictions sont plus fortes ?
annees_covid = ventes.annee.isin(config.ANNEES_ATTENDUES)
pays_jour = (ventes[(ventes.canal == "Store") & annees_covid].groupby(["pays", "date"])
             .agg(ca=("montant_ventes", "sum"), severite=("severite_covid", "first")).reset_index())
# On centre le CA par pays (CA du jour ÷ CA moyen du pays) pour comparer des pays de tailles différentes.
pays_jour["ca_relatif"] = pays_jour.ca / pays_jour.groupby("pays").ca.transform("mean")
correlation = pays_jour.ca_relatif.corr(pays_jour.severite)
print(f"E1. Corrélation entre sévérité COVID et CA relatif des magasins (pays × jour, {config.ANNEES_ATTENDUES}) : {correlation:+.3f}")
print("    Une vraie activité de 2020 montrerait une corrélation nettement négative.")

# ---------------------------------------------------------------------------
# E2. La baisse de 2020 par rapport à 2019 peut-elle venir du COVID ?
# ---------------------------------------------------------------------------
# Avec 2019 comme référence, les magasins perdent beaucoup en 2020. Une lecture rapide dirait :
# c'est la pandémie. Deux vérifications simples :
#   a) la baisse existe-t-elle déjà en janvier et février 2020, AVANT toute restriction ?
#   b) la vente en ligne récupère-t-elle ce que les magasins perdent ?
magasins = ventes[ventes.canal == "Store"]
par_mois = magasins.groupby(["mois", "annee"]).montant_ventes.sum().unstack("annee")
severite_mois = ventes.groupby(["mois", "annee"]).severite_covid.mean().unstack("annee")
reference, premiere = config.ANNEE_REFERENCE, min(config.ANNEES_ATTENDUES)
comparaison = pd.DataFrame({
    f"ca_magasins_{reference}": par_mois[reference], f"ca_magasins_{premiere}": par_mois[premiere],
    "evolution": par_mois[premiere] / par_mois[reference] - 1,
    f"indice_covid_{premiere}": severite_mois[premiere]})
print(f"\nE2a. Ventes des magasins, {premiere} comparé à {reference}, mois par mois :")
display(comparaison.style.format({f"ca_magasins_{reference}": "{:,.0f}", f"ca_magasins_{premiere}": "{:,.0f}",
                                  "evolution": "{:+.1%}", f"indice_covid_{premiere}": "{:.0f}"}))

par_canal = ventes.groupby(["annee", "canal"]).montant_ventes.sum().unstack("canal")
print("E2b. Chiffre d'affaires par canal et par année :")
display(par_canal.style.format("{:,.0f}"))
gain_en_ligne = par_canal.loc[premiere, "Online"] - par_canal.loc[reference, "Online"]
perte_magasins = par_canal.loc[reference, "Store"] - par_canal.loc[premiere, "Store"]
print(f"     Entre {reference} et {premiere} : les magasins perdent {perte_magasins:,.0f} $, ".replace(",", " ")
      + f"la vente en ligne gagne {gain_en_ligne:,.0f} $ ({gain_en_ligne / perte_magasins:.0%} de la perte).".replace(",", " "))

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : chiffre d'affaires des magasins (canal "Store") et sévérité COVID, par mois
# ---------------------------------------------------------------------------
mensuel = (ventes[ventes.canal == "Store"].groupby(ventes.date.dt.to_period("M"))
           .agg(ca=("montant_ventes", "sum")))
mensuel["severite"] = ventes.groupby(ventes.date.dt.to_period("M")).severite_covid.mean()
positions = range(len(mensuel))

fig, ax = plt.subplots(figsize=(13, 4.6))
ax.bar(positions, mensuel.ca, color=viz.PRINCIPALE, width=0.75)
viz.format_dollars(ax, axe="y")
ax.set_ylabel("Chiffre d'affaires des magasins")
ax.set_xticks(list(positions), [str(m) for m in mensuel.index], rotation=45, ha="right")

# Deuxième axe (à droite) pour l'indice de sévérité, en couleur d'accent
ax2 = ax.twinx()
ax2.plot(positions, mensuel.severite, color=viz.ACCENT, linewidth=2.5, marker="o")
ax2.set_ylim(0, 100)
ax2.set_ylabel("Indice de sévérité COVID (0 à 100)", color=viz.ACCENT)
ax2.grid(False)
ax2.spines["right"].set_visible(True)
viz.titre(ax, "La baisse des ventes en magasin précède les restrictions COVID",
          "Barres : chiffre d'affaires mensuel des magasins · ligne orange : indice de sévérité moyen (0 en 2019)")
print("Figure enregistrée :", viz.enregistrer(fig, "03_covid_et_ventes"))
plt.show()

### Ce que ces cinq tests démontrent

1. **Pas de client, pas de ticket.** `id_vente` ne numérote pas des passages en caisse mais des lignes de cumul : une seule ligne par jour, boutique, produit, promotion et canal. La table n'est pas une table de faits **transactionnelle** (le journal des ventes), c'est une table de faits **agrégée**, du type de celles qu'on génère après la clôture de la journée. Le journal d'origine n'est pas fourni. Le panier moyen, le nombre de clients, l'heure d'achat et la fidélité ne sont pas calculables, et un « nombre de ventes » compté sur cette table est un nombre de lignes, pas un nombre d'achats.
2. **Aucun produit vendu à l'unité**, même à plus de 1 000 $. La quantité minimale d'une ligne est de 2, et ce minimum ne concerne que *Home Appliances* et *Games and Toys* : les six autres catégories ne descendent jamais sous 4 unités. Le plancher change d'une année à l'autre pour toute une catégorie à la fois (4 en 2019, 8 en 2020, 6 en 2021 pour les six catégories) : c'est un paramètre de génération, pas un comportement d'achat. La quantité ne prend que 96 valeurs sur 715 291 lignes, et les 8 plus fréquentes couvrent 81 % des lignes. Même pour un cumul journalier, c'est invraisemblable : un vrai magasin a des jours où il ne vend qu'un seul exemplaire d'un produit cher.
3. **Aucun prix ni aucun coût ne change en trois ans**, sur les 2 244 produits vendus au moins deux années. Ni inflation, ni baisse du prix de l'électronique.
4. **L'activité est répartie de façon presque uniforme sur les jours de la semaine**, sans creux ni pic marqué (0,2 point d'écart entre le jour le plus fort et le plus faible).
5. **Aucune réaction au COVID-19.** C'est attendu : les ventes datent en réalité de 2007 à 2009 et ont été décalées de 12 ans. Elles n'ont jamais connu les confinements. Avec 2019 dans la table, ce point demande plus de soin qu'avant, parce que les magasins perdent 40 % de leurs ventes entre 2019 et 2020 et qu'on pourrait y voir l'effet de la pandémie. Trois faits l'excluent :
   - la baisse est déjà de 43 % en **janvier 2020** et de 35 % en février, quand l'indice de sévérité vaut 1 puis 10 sur 100. Le niveau des ventes ne bouge pas quand l'indice passe de 52 en mars à 82 en avril (28,3 M$ puis 28,5 M$) ;
   - **la vente en ligne ne prend pas le relais** : elle gagne 23 M$ quand les magasins en perdent 209, soit 11 % de la perte. Un confinement aurait déplacé les achats, pas supprimé ;
   - à l'intérieur de 2020 et 2021, un pays ne vend pas moins les jours où ses restrictions sont plus fortes (corrélation de +0,02).

   La chute des magasins entre 2019 et 2020 est donc un fait de la base Contoso, à expliquer par ses propres variables (assortiment, publicité, promotions), et non par le contexte sanitaire. C'est l'objet des notebooks d'analyse.

Un sixième indice s'ajoute à ces cinq tests : les remises portant sur plus d'unités que la vente (section 7, règle R5), qui s'expliquent par un tirage de la quantité remisée sans lien avec la quantité vendue.

**Conclusion** : le jeu de données est **généré par des règles** (base de démonstration Contoso). Il est cohérent sur le plan comptable, mais il ne reflète pas une activité commerciale réelle.

**Conséquences pour la suite du projet**

- Les relations qu'on trouvera (effet des promotions, retours) décrivent les règles de génération, pas le comportement de vrais clients. Il faudra le dire dans le rapport.
- L'indice COVID et les jours fériés restent dans la table comme variables de contexte, mais on s'attend à **ne trouver aucun effet**. Un résultat nul, expliqué, est un résultat valable.
- Q5 (optimisation des prix) ne peut porter que sur les **remises**, puisque les prix catalogue ne varient jamais.
- Les analyses par client sont à écarter explicitement.

In [ ]:
noter("11. Nature", "Jeu de données généré (5 tests : clients, quantités, prix, régularité, COVID)", 5,
      "Information", "Présenter cette limite dans L3 et dans le rapport final", total=5)

## 12. Synthèse : registre des anomalies

Le registre rassemble tous les contrôles. Les lignes **À corriger** ont un traitement évident. Les lignes **À décider** demandent un choix de l'équipe : ce choix doit être écrit et justifié dans le livrable L3, puis appliqué dans `04_nettoyage`.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Registre complet
# ---------------------------------------------------------------------------
registre_df = pd.DataFrame(registre)

# Résumé par niveau de gravité
ordre = ["À décider", "À corriger", "Information", "OK"]
print(registre_df.gravite.value_counts().reindex(ordre).fillna(0).astype(int).to_string())

# ---------------------------------------------------------------------------
# 2. Ce qui demande une action ou une attention : tout sauf les contrôles "OK"
# ---------------------------------------------------------------------------
a_traiter = (registre_df[registre_df.gravite != "OK"]
             .assign(rang=lambda d: d.gravite.map({g: i for i, g in enumerate(ordre)}))
             .sort_values(["rang", "nb"], ascending=[True, False])
             .drop(columns="rang"))
pd.set_option("display.max_colwidth", 140)
display(a_traiter.style.format({"nb": "{:,.0f}", "part": "{:.2%}"}).hide(axis="index"))

In [ ]:
# ---------------------------------------------------------------------------
# 3. Enregistrement du rapport de diagnostic (pièce du livrable L3)
# ---------------------------------------------------------------------------
# Un classeur Excel, une feuille par sujet : il peut être lu sans ouvrir le notebook.
chemin_rapport = config.L3 / "diagnostic_qualite.xlsx"
with pd.ExcelWriter(chemin_rapport) as classeur:
    registre_df.to_excel(classeur, sheet_name="Registre", index=False)
    structure.to_excel(classeur, sheet_name="Structure")
    manquantes.to_excel(classeur, sheet_name="Manquantes")
    texte.to_excel(classeur, sheet_name="Texte")
    tableau_r4.to_excel(classeur, sheet_name="Regle_cout_total")
    logique_par_annee.to_excel(classeur, sheet_name="Cout_total_par_annee")
    detail_hors.to_excel(classeur, sheet_name="Promo_hors_periode", index=False)
    activite.to_excel(classeur, sheet_name="Activite_boutiques", index=False)
    extremes_tab.to_excel(classeur, sheet_name="Extremes")
    jumelles.to_excel(classeur, sheet_name="Promo_jumelles", index=False)
    prix_par_produit.to_excel(classeur, sheet_name="Prix_par_produit")
    covid.to_excel(classeur, sheet_name="Covid_trimestre")
print("Rapport enregistré :", chemin_rapport)

# ---------------------------------------------------------------------------
# 4. Contrôles bloquants : si l'un échoue, on ne passe pas au nettoyage
# ---------------------------------------------------------------------------
assert ventes.id_vente.is_unique, "Identifiants de vente en double."
assert vides_obligatoires == 0, "Vides dans des colonnes obligatoires."
assert not mauvaises_dates and not mauvais_nombres, "Types incorrects."
print("Contrôles bloquants : OK. La table peut passer à 04_nettoyage.")

### Décisions prises par l'équipe

| Constat | Décision | Où |
|---|---|---|
| Espaces en trop dans les colonnes texte | Les retirer | `04_nettoyage` |
| `continent` (une seule valeur) et `date_fermeture` (entièrement vide) | Supprimer ces deux colonnes de la table propre | `04_nettoyage` |
| Ventes rattachées à une promotion hors de sa période (29 712 lignes de janvier 2019 et janvier 2021) | Garder les ventes et leurs remises, ajouter l'indicateur `promo_hors_periode` | `04_nettoyage` |
| Ventes datées avant l'ouverture du magasin (189 lignes de 2019, six magasins) | Garder les lignes (elles font partie des totaux du plan), ajouter l'indicateur `avant_ouverture` (règle N8) ; comparer les magasins sur leurs mois d'activité après ouverture | `04_nettoyage` |
| Grandes quantités (au-dessus de la borne IQR) | Garder, ajouter l'indicateur `grande_quantite` | `04_nettoyage` |
| Quantité remisée supérieure à la quantité vendue (2 331 lignes : 1 945 en 2019, 386 en 2021) | Ramener la quantité remisée à la quantité vendue, sans modifier la remise ni le montant des ventes ; ajouter l'indicateur `remise_corrigee` ; garder les quantités d'origine dans le journal | `04_nettoyage` |
| Effectif vide pour une boutique | Imputer par la médiane des magasins du même pays et de même surface ; ajouter l'indicateur `effectif_impute` (règle N7). La masse salariale ne permet pas de le déduire (`05b_charges`) | `04_nettoyage` |
| Montant des ventes | Inchangé : c'est le chiffre d'affaires brut | aucun traitement |
| Coût total calculé de trois façons selon les lignes | Ne pas modifier la colonne ; recalculer coût brut, coût du retour et coût net | `05_variables` |
| Taux de remise annoncé presque jamais atteint, mécanisme inconnu | Ne rien corriger (les montants sont réels) ; calculer le taux de remise effectif de chaque ligne | `05_variables` |
| Jeu de données généré, table agrégée sans client | Limite à écrire dans le rapport | L3 et rapport final |
| Indice COVID sans effet ; baisse des magasins de 2020 antérieure aux restrictions | Garder la variable de contexte ; ne pas attribuer la baisse de 2020 à la pandémie | L3 et notebooks d'analyse |

**Suite** : `04_nettoyage` applique les règles validées par l'équipe et écrit la table propre dans `data/clean`. `05_variables` calcule ensuite les variables d'analyse (marge en valeur, taux de remise effectif, taux de retour).